# 实验二：房源采集与 MySQL 存储

- 姓名：王永亮　学号：112404260117　班级：数据1241
- 数据来源：安居客教学站 http://www.bspider.top/anjuke/
- 目标表：`smartpark_lab.houseinfo`（8 个字段：title / house / address / struct / area / floor / maketime / price）

**本 Notebook 分两个阶段运行：**

1. 第二阶段：采集 → 解析 → 翻页 → 保存 `data/houseinfo.csv`
2. 第三阶段：读取 `data/houseinfo.csv` → 写入 MySQL（**不重新爬网页**）

第一阶段（Docker 启动、建表）在 `mysql-console.bat` 里完成，不在本文件里。

> 注意：请求前先看第二节的输出。若标题出现“访问验证”，说明站点拦截了，改用教师提供的网页快照（放到 `lab02/html/`），并如实记录为离线回放。


## 0. 环境准备

缺少依赖时运行下面的安装单元（BeautifulSoup 安装名是 `beautifulsoup4`，导入名是 `bs4`）。

In [ ]:
%pip install requests beautifulsoup4 pymysql

In [1]:
import csv, time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print("工作目录：", Path.cwd())

工作目录： /home/pwuser/workspace/lab02


## 第二阶段 1. 获取第一页

教学站的 chunked 传输经常中断，`requests.get()` 会中途 `Read timed out`；
偶发还会返回 6 KB 左右的「访问验证」页（状态码仍是 200）。
所以下面把请求封装成 `fetch()`：失败重试、http/https 双协议回退、并用页面长度判断是否拿到真页面。

In [2]:
import os, time
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup

BASE = "http://www.bspider.top/anjuke/"
ALLOW_HOST = "www.bspider.top"      # 只允许在教学站内跳转
HTML_DIR = "html"                   # 每页原始 HTML 存为快照，便于离线回放与核对
os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
}
MIN_LEN = 5000      # 访问验证页约 6 KB，低于此长度判为异常
MAX_PAGE = 30       # 安全页数上限
SLEEP = 1.5         # 每页之间至少等待 1 秒


def fetch(url, tries=5):
    """请求网页：失败重试 + 双协议回退 + 长度校验。返回 (html, 重试次数)。"""
    last = ""
    for i in range(tries):
        for u in (url, url.replace("http://", "https://")):
            try:
                r = requests.get(u, headers=HEADERS, timeout=(10, 40))
                r.raise_for_status()          # 非 2xx 直接报错，不把错误页当房源页
                r.encoding = r.apparent_encoding
                if len(r.text) < MIN_LEN:
                    last = f"页面过短({len(r.text)}字节，疑似访问验证页)"
                    continue
                return r.text, i + 1
            except Exception as e:
                last = f"{type(e).__name__}: {str(e)[:60]}"
        time.sleep(1)
    raise RuntimeError(f"抓取失败 {url} -> {last}")


def save_snapshot(html, page_no):
    path = os.path.join(HTML_DIR, f"page_{page_no:02d}.html")
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write(html)
    return path


# ---- 请求第一页，自己检查返回内容 ----
page_no = 1
current_url = BASE
html, tries = fetch(current_url)
print("status_code : 200（fetch 内部已 raise_for_status）")
print("重试次数    :", tries)
print("页面长度    :", len(html))

soup = BeautifulSoup(html, "html.parser")
print("网页标题    :", soup.title.get_text(strip=True))
print("div.property 命中 :", len(soup.select("div.property")))
print("快照已保存  :", save_snapshot(html, page_no))

status_code : 200（fetch 内部已 raise_for_status）
重试次数    : 1
页面长度    : 280794
网页标题    : 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
div.property 命中 : 60
快照已保存  : html/page_01.html


### 拿到的是访问验证页或超时，怎么办

- `len(html) < 5000` 判为验证页，`fetch()` 会自动重试；5 次仍失败就抛错，不把验证页当房源页解析。
- 每次成功抓取都把原始 HTML 写入 `html/page_NN.html`。即使之后站点再次不可用，也能从这些快照离线回放、继续做字段解析和 CSV。
- 站点彻底不可用时，把教师提供的 HTML 放进 `lab02/html/`，运行下一格读取。

In [3]:
# 离线回放分支：从 html/ 下的快照读取（站点不可用时使用）
snapshots = sorted(f for f in os.listdir(HTML_DIR) if f.lower().endswith(".html"))
print("html/ 中的快照：", snapshots)

USE_SNAPSHOT = False        # 改为 True 即离线回放，不联网
if USE_SNAPSHOT and snapshots:
    with open(os.path.join(HTML_DIR, snapshots[0]), encoding="utf-8") as f:
        html = f.read()
    soup = BeautifulSoup(html, "html.parser")
    print("已从快照载入：", snapshots[0], "| div.property 命中：", len(soup.select("div.property")))

html/ 中的快照： ['page_01.html', 'page_02.html', 'page_03.html', 'page_04.html', 'page_05.html', 'page_06.html', 'page_07.html', 'page_08.html', 'page_09.html', 'page_10.html']


## 第二阶段 2. 分析网页，提取 8 个字段

先确认父容器 `div.property` 命中条数，再打印第一条的属性组原文，确认「户型/面积/朝向/楼层/年代」的实际顺序。

In [4]:
rows = soup.select("div.property")      # select() 返回列表，每项对应一条房源
print("本页房源条数 :", len(rows))

row = rows[0]
infos = [p.get_text(strip=True)
         for p in row.select("div.property-content-info p.property-content-info-text")]
print("第一条的属性组原文：", infos)
print("第二条的属性组原文：",
      [p.get_text(strip=True)
       for p in rows[1].select("div.property-content-info p.property-content-info-text")] if len(rows) > 1 else "（本页不足2条）")

本页房源条数 : 60
第一条的属性组原文： ['3室2厅1卫', '89.9㎡', '南北', '低层(共27层)', '2019年建造']
第二条的属性组原文： ['3室2厅2卫', '94.5㎡', '东南', '低层(共28层)', '2008年建造']


### 在一条房源内找字段（相对选择器）

`select_one()` 只在**这一条房源内部**查找，不必保留从页面根节点开始的长路径。取值要点：

| 字段 | 选择器 | 取值要点 |
|---|---|---|
| title | `h3.property-content-title-name` | 取 `title` 属性，它是未被省略号截断的完整标题 |
| house | `p.property-content-info-comm-name` | 楼盘名 |
| address | `p.property-content-info-comm-address` | 文字分散在多个 `span`，`get_text(strip=True)` **不在子标签间插空格** |
| struct | `p.property-content-info-attribute` | 户型，如 `3室2厅1卫` |
| area | 属性组中 `endswith("㎡")` 的项 | 保留单位 |
| floor | 属性组中 `含"层"` 的项 | 朝向是「南北/东南」，不含"层"，不会取错 |
| maketime | 属性组中 `endswith("年建造")` 的项 | 缺失时填空字符串 |
| price | `p.property-price-average` | 单价 `元/㎡`；`property-price-total` 是总价「148万」，不能取 |

按内容特征判断而不写死下标，页面少一项或多一项时都不会取错。

In [5]:
def parse_row(row):
    """一条房源 -> [title, house, address, struct, area, floor, maketime, price]"""
    # 1) 标题：h3 的 title 属性保存了未被省略号截断的完整文字
    h3 = row.select_one("h3.property-content-title-name")
    title = ""
    if h3 is not None:                       # select_one 找不到时返回 None，先判断再用
        title = (h3.get("title") or "").strip() or h3.get_text(strip=True)

    # 2) 楼盘
    node = row.select_one("p.property-content-info-comm-name")
    house = node.get_text(strip=True) if node is not None else ""

    # 3) 地址：多个 span 拼接，不主动插入空格
    node = row.select_one("p.property-content-info-comm-address")
    address = node.get_text(strip=True) if node is not None else ""

    # 4) 属性组：顺序可能变化，按内容特征判断
    texts = [p.get_text(strip=True)
             for p in row.select("div.property-content-info p.property-content-info-text")]
    attr = row.select_one("p.property-content-info-attribute")
    struct = attr.get_text(strip=True) if attr is not None else ""
    rest = [t for t in texts if t != struct]        # 去掉户型项，剩下 面积/朝向/楼层/年代

    area     = next((t for t in rest if t.endswith("㎡")), "")       # 建筑面积
    floor    = next((t for t in rest if "层" in t), "")              # 楼层（朝向不含"层"）
    maketime = next((t for t in rest if t.endswith("年建造")), "")   # 缺失时为空字符串

    # 5) 单价：取 average（元/㎡），不是 total（万元总价）
    node = row.select_one("p.property-price-average")
    price = node.get_text(strip=True) if node is not None else ""

    return [title, house, address, struct, area, floor, maketime, price]


FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

first = parse_row(rows[0])
for k, v in zip(FIELDS, first):
    print(f"{k:9s} = {v!r}")

title     = '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
house     = '万科金域学府翰林菁华里'
address   = '南岸学府大道二塘路59号'
struct    = '3室2厅1卫'
area      = '89.9㎡'
floor     = '低层(共27层)'
maketime  = '2019年建造'
price     = '16465元/㎡'


In [6]:
# 校对第二条和最后一条：面积、楼层、年代、单价是否取对
for idx in [1, len(rows) - 1]:
    if idx >= len(rows) or idx == 0:
        continue
    print(f"---- 第 {idx + 1} 条 ----")
    for k, v in zip(FIELDS, parse_row(rows[idx])):
        print(f"  {k:9s} = {v!r}")

---- 第 2 条 ----
  title     = '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住'
  house     = '两代一家'
  address   = '渝北渝北黄泥磅紫荆路29号'
  struct    = '3室2厅2卫'
  area      = '94.5㎡'
  floor     = '低层(共28层)'
  maketime  = '2008年建造'
  price     = '14815元/㎡'
---- 第 60 条 ----
  title     = '精装修 电梯房 4室2厅 91平'
  house     = '华润凯旋天地'
  address   = '南岸弹子石腾龙大道45号'
  struct    = '4室2厅2卫'
  area      = '91.3㎡'
  floor     = '低层(共32层)'
  maketime  = '2015年建造'
  price     = '15886元/㎡'


## 第二阶段 3. 采集下一页

`a.next.next-active` 是**可用**的下一页按钮；末页时这个 class 会消失。
`get_text()` 取到的是「下一页」文字，`get("href")` 取到的才是地址。

In [7]:
nxt = soup.select_one("a.next.next-active")
print("下一页节点 :", nxt.get_text(strip=True) if nxt else None)
print("href 原值  :", nxt.get("href") if nxt else None)
print("class      :", nxt.get("class") if nxt else None)

if nxt:
    next_url = urljoin(current_url, nxt.get("href"))     # href 只有 ?page=2，需要补全
    print("urljoin 补全后 :", next_url)
    print("主机是否仍为教学站 :", urlparse(next_url).netloc == ALLOW_HOST)

下一页节点 : 下一页
href 原值  : ?page=2
class      : ['next', 'next-active']
urljoin 补全后 : http://www.bspider.top/anjuke/?page=2
主机是否仍为教学站 : True


### 分页循环

停止条件分三种，不能混为一谈：

- **正常结束**：分页区域仍在，但 `a.next` 没有 `next-active`（下一页按钮已禁用）；
- **结构异常**：连 `a.next` 都找不到，或本页房源列表为空 —— 记为异常，不算采集完成；
- **中止**：地址越出教学站、地址重复、或达到安全页数上限 `MAX_PAGE`。

In [8]:
def crawl(start_url, max_pages=MAX_PAGE, use_snapshot=False):
    """翻页采集。结果列表是函数内的局部变量，最后 return，不用全局变量。"""
    records = []                 # 局部变量：每次调用都从空列表开始，不会被上次运行污染
    visited = set()              # 防止重复访问同一个地址
    page, stop, url = 1, "", start_url

    while page <= max_pages:
        if use_snapshot:                                   # 离线回放：读本地快照
            path = os.path.join(HTML_DIR, f"page_{page:02d}.html")
            if not os.path.exists(path):
                stop = "离线回放结束：快照已读完"
                print(stop)
                break
            with open(path, encoding="utf-8") as f:
                page_html = f.read()
            tries = 0
        else:
            page_html, tries = fetch(url)
            save_snapshot(page_html, page)

        soup_i = BeautifulSoup(page_html, "html.parser")
        rows_i = soup_i.select("div.property")
        if not rows_i:                                     # 空列表按错误处理
            stop = f"结构异常：第{page}页房源列表为空"
            print(stop)
            break

        records.extend(parse_row(r) for r in rows_i)
        print(f"第{page:02d}页 | URL={url} | 重试{tries}次 | 本页{len(rows_i)}条 | 累计{len(records)}条")

        nxt = soup_i.select_one("a.next.next-active")
        if nxt is None:                                    # 没有可用的下一页
            stop = ("正常结束：分页区域存在，下一页按钮已禁用"
                    if soup_i.select_one("a.next") is not None
                    else "结构异常：找不到下一页按钮，分页区域可能已消失")
            print(stop)
            break

        href = nxt.get("href")
        next_url = urljoin(url, href)
        print(f"        href={href!r} -> urljoin -> {next_url}")

        if urlparse(next_url).netloc != ALLOW_HOST:        # 越出教学站
            stop = f"异常中止：链接越出教学站 {next_url}"
            print(stop)
            break
        if next_url in visited:                            # 地址重复
            stop = f"异常中止：地址重复 {next_url}"
            print(stop)
            break

        visited.add(next_url)
        url = next_url
        page += 1
        time.sleep(SLEEP)                                  # 至少等待 1 秒

    if page > max_pages:
        stop = f"达到安全页数上限 {max_pages} 页，记为中止"
        print(stop)
    return records, page, stop


records, pages, stop_reason = crawl(BASE, use_snapshot=USE_SNAPSHOT)
print("\n成功采集页数 :", pages)
print("停止原因     :", stop_reason)
print("去重前条数   :", len(records))

第01页 | URL=http://www.bspider.top/anjuke/ | 重试1次 | 本页60条 | 累计60条
        href='?page=2' -> urljoin -> http://www.bspider.top/anjuke/?page=2


第02页 | URL=http://www.bspider.top/anjuke/?page=2 | 重试1次 | 本页60条 | 累计120条
        href='?page=3' -> urljoin -> http://www.bspider.top/anjuke/?page=3


第03页 | URL=http://www.bspider.top/anjuke/?page=3 | 重试1次 | 本页60条 | 累计180条
        href='?page=4' -> urljoin -> http://www.bspider.top/anjuke/?page=4


第04页 | URL=http://www.bspider.top/anjuke/?page=4 | 重试1次 | 本页60条 | 累计240条
        href='?page=5' -> urljoin -> http://www.bspider.top/anjuke/?page=5


第05页 | URL=http://www.bspider.top/anjuke/?page=5 | 重试1次 | 本页60条 | 累计300条
        href='?page=6' -> urljoin -> http://www.bspider.top/anjuke/?page=6


第06页 | URL=http://www.bspider.top/anjuke/?page=6 | 重试1次 | 本页60条 | 累计360条
        href='?page=7' -> urljoin -> http://www.bspider.top/anjuke/?page=7


第07页 | URL=http://www.bspider.top/anjuke/?page=7 | 重试2次 | 本页60条 | 累计420条
        href='?page=8' -> urljoin -> http://www.bspider.top/anjuke/?page=8


第08页 | URL=http://www.bspider.top/anjuke/?page=8 | 重试1次 | 本页60条 | 累计480条
        href='?page=9' -> urljoin -> http://www.bspider.top/anjuke/?page=9


第09页 | URL=http://www.bspider.top/anjuke/?page=9 | 重试1次 | 本页60条 | 累计540条
        href='?page=10' -> urljoin -> http://www.bspider.top/anjuke/?page=10


第10页 | URL=http://www.bspider.top/anjuke/?page=10 | 重试1次 | 本页60条 | 累计600条
正常结束：分页区域存在，下一页按钮已禁用

成功采集页数 : 10
停止原因     : 正常结束：分页区域存在，下一页按钮已禁用
去重前条数   : 600


## 第二阶段 4. 保存和核对 CSV

按 8 个字段**完全相同**去重，再用 `csv.writer` 保存：
编码 `utf-8-sig`（Excel 打开不乱码）、`newline=""`（不产生空行）。

In [2]:
# ---- 按 8 个字段完全相同去重 ----
unique, seen = [], set()
for rec in records:
    key = tuple(rec)                      # 一条房源 = 8 个字段组成的元组
    if key not in seen:
        seen.add(key)
        unique.append(rec)
print("去重前条数 :", len(records))
print("去重后条数 :", len(unique))
print("完全相同被去掉的重复记录 :", len(records) - len(unique))

# ---- 保存 CSV：一行一套房源 ----
csv_path = os.path.join("data", "houseinfo.csv")
with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:      # 编码 + newline
    w = csv.writer(f)
    w.writerow(FIELDS)                    # 表头：8 个字段的字段名，顺序固定
    w.writerows(unique)                   # 每条记录恰好 8 项
print("已保存 :", csv_path, "| 数据行数 :", len(unique))

去重前条数 : 600
去重后条数 : 498
完全相同被去掉的重复记录 : 102
已保存 : data/houseinfo.csv | 数据行数 : 498


In [3]:
# ---- 回读核对 ----
with open(csv_path, encoding="utf-8-sig", newline="") as f:
    rows_csv = list(csv.DictReader(f))
print("CSV 数据行数（不含表头） :", len(rows_csv))
print("文件总行数（含表头）     :", sum(1 for _ in open(csv_path, encoding="utf-8-sig")))
print("表头                     :", list(rows_csv[0].keys()))

# 逐字段抽查：第一条、中间一条、最后一条
for pos in (0, len(rows_csv) // 2, len(rows_csv) - 1):
    print(f"\n---- 第 {pos + 1} 条 ----")
    for k in FIELDS:
        print(f"  {k:9s} = {rows_csv[pos][k]!r}")

CSV 数据行数（不含表头） : 498
文件总行数（含表头）     : 499
表头                     : ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']

---- 第 1 条 ----
  title     = '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
  house     = '万科金域学府翰林菁华里'
  address   = '南岸学府大道二塘路59号'
  struct    = '3室2厅1卫'
  area      = '89.9㎡'
  floor     = '低层(共27层)'
  maketime  = '2019年建造'
  price     = '16465元/㎡'

---- 第 250 条 ----
  title     = '精装修洋房 位置安静 全新装修 随时看房'
  house     = '协信城立方玺院'
  address   = '沙坪坝大学城大学城南二路'
  struct    = '3室3厅2卫'
  area      = '95.8㎡'
  floor     = '中层(共8层)'
  maketime  = '2016年建造'
  price     = '15658元/㎡'

---- 第 498 条 ----
  title     = '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售'
  house     = '恒大照母山山水城'
  address   = '渝北照母山恒康路9号'
  struct    = '3室2厅1卫'
  area      = '90.2㎡'
  floor     = '低层(共32层)'
  maketime  = '2015年建造'
  price     = '15530元/㎡'


In [1]:
# ---- 交作业自检：只读 CSV 文件本身，不依赖前面的采集 ----
import csv
from pathlib import Path

p = Path("data") / "houseinfo.csv"
with p.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

print("文件 :", p.resolve())
print("表头 :", list(rows[0].keys()))
print("数据行数（不含表头） :", len(rows))
print()
print("前 3 行 :")
for r in rows[:3]:
    print("  " + " | ".join(r[k] for k in r))

文件 : /home/pwuser/workspace/lab02/data/houseinfo.csv
表头 : ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
数据行数（不含表头） : 498

前 3 行 :
  万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器 | 万科金域学府翰林菁华里 | 南岸学府大道二塘路59号 | 3室2厅1卫 | 89.9㎡ | 低层(共27层) | 2019年建造 | 16465元/㎡
  黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住 | 两代一家 | 渝北渝北黄泥磅紫荆路29号 | 3室2厅2卫 | 94.5㎡ | 低层(共28层) | 2008年建造 | 14815元/㎡
  鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售 | 大雅云居山 | 渝北鸳鸯龙展路59号 | 3室2厅1卫 | 88㎡ | 低层(共24层) | 2016年建造 | 16487元/㎡


## 第三阶段：MySQL 数据存储

读取第二阶段保存的 `data/houseinfo.csv` 写入数据库，**不重新爬取网页**。

| 连接参数 | 值 |
|---|---|
| host | `mysql`（容器内服务名，不能写 localhost） |
| port | `3306` |
| database | `smartpark_lab` |
| user | `student` |
| password | `123456`（课程教学库统一密码） |
| charset | `utf8mb4` |
| autocommit | `False`，事务由程序 `commit()` |

要点：SQL 里用 `%s` 占位，实际值通过第二个参数传入，**不要用字符串拼接或 f-string**。
这样 8 个字段的值由驱动负责转义，标题里的引号、空格不会破坏 SQL。

In [1]:
import pymysql

conn = pymysql.connect(
    host="mysql", port=3306,
    user="student", password="123456",
    database="smartpark_lab",
    charset="utf8mb4",
    autocommit=False,              # 不自动提交，由程序决定何时 commit
)

# 先查一条数据，确认连接、库名和权限都正常
with conn.cursor() as cur:
    cur.execute("SELECT VERSION(), DATABASE(), CURRENT_USER()")
    print("版本 / 当前库 / 当前用户 :", cur.fetchone())
    cur.execute("SELECT COUNT(*) FROM houseinfo")
    print("入库前 houseinfo 条数     :", cur.fetchone()[0])

conn.close()
print("连接正常，测试完已关闭")

版本 / 当前库 / 当前用户 : ('8.4.11', 'smartpark_lab', 'student@%')
入库前 houseinfo 条数     : 0
连接正常，测试完已关闭


### 1. 读取 CSV，整理入库参数

`csv.DictReader` 按表头取列，所以字段顺序由 `FIELDS` 决定，不依赖 CSV 列的物理顺序。
每条记录按 `FIELDS` 取出 8 个值，正好对应 INSERT 里 8 个 `%s` 的位置。

In [2]:
import csv
from pathlib import Path

FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]
CSV_PATH = Path("data") / "houseinfo.csv"

with CSV_PATH.open(encoding="utf-8-sig", newline="") as f:     # utf-8-sig 去掉 BOM
    rows_csv = list(csv.DictReader(f))

print("文件         :", CSV_PATH.resolve())
print("表头         :", list(rows_csv[0].keys()))
print("表头与 FIELDS 一致 :", list(rows_csv[0].keys()) == FIELDS)
print("数据行数     :", len(rows_csv))

# 每条记录 -> 8 个参数，顺序与 INSERT 的字段顺序完全一致
csv_rows = [[r[k] for k in FIELDS] for r in rows_csv]
print("每条记录的参数个数 :", {len(r) for r in csv_rows})
print("第一条的 8 个参数  :")
for name, val in zip(FIELDS, csv_rows[0]):
    print(f"  {name:9s} <- {val!r}")

文件         : /home/pwuser/workspace/lab02/data/houseinfo.csv
表头         : ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
表头与 FIELDS 一致 : True
数据行数     : 498
每条记录的参数个数 : {8}
第一条的 8 个参数  :
  title     <- '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
  house     <- '万科金域学府翰林菁华里'
  address   <- '南岸学府大道二塘路59号'
  struct    <- '3室2厅1卫'
  area      <- '89.9㎡'
  floor     <- '低层(共27层)'
  maketime  <- '2019年建造'
  price     <- '16465元/㎡'


### 2. 批量入库

- `executemany(sql, to_insert)`：一次提交多条记录，每条都是 8 个值；
- `commit()`：成功才提交，提交后 MySQL 终端里才能查到新数据；
- `rollback()`：出错时撤销本次事务里尚未提交的修改；
- `finally`：无论成功失败都先关游标、再关连接；
- 重复运行：先 `SELECT` 出库内已有的 8 字段组合，用集合比对，只插入尚不存在的记录。
  自增 `id` 挡不住重复，所以这里按 8 个字段比较，而不是清空表重来。

In [3]:
import pymysql

INSERT_SQL = """
INSERT INTO houseinfo
    (title, house, address, struct, area, floor, maketime, price)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

conn = pymysql.connect(
    host="mysql", port=3306, user="student", password="123456",
    database="smartpark_lab", charset="utf8mb4",
    autocommit=False,                       # 事务由程序提交
)
cur = None
try:
    cur = conn.cursor()

    # ① 库内已有的 8 字段组合 -> 集合，用于判断“是否已存在”
    cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
    existing = {tuple(x) for x in cur.fetchall()}

    # ② 只保留库内没有的记录，重复运行不会重复插入
    to_insert = [tuple(r) for r in csv_rows if tuple(r) not in existing]
    print(f"CSV {len(csv_rows)} 条 | 库内已有 {len(existing)} 条 | 本次待插入 {len(to_insert)} 条")

    if to_insert:
        affected = cur.executemany(INSERT_SQL, to_insert)    # 批量执行，每条 8 个参数
        conn.commit()                                        # ③ 成功：提交事务
        print("executemany 返回受影响行数 =", affected, "（是本次写入的行数，不是表的总条数）")
    else:
        print("库内已存在同一批数据，本次不插入")
except Exception as e:
    conn.rollback()                                          # ④ 失败：撤销未提交的修改
    print("写入失败，已回滚：", type(e).__name__, e)
finally:
    if cur is not None:
        cur.close()                                          # ⑤ 先关游标
    conn.close()                                             # ⑥ 再关连接
    print("游标与连接已关闭")

CSV 498 条 | 库内已有 0 条 | 本次待插入 498 条
executemany 返回受影响行数 = 498 （是本次写入的行数，不是表的总条数）
游标与连接已关闭


### 3. 入库后核对

条数应等于去重后的 CSV 行数（498）；再把库内 8 字段与 CSV 做集合比较，确认内容逐个相同。

In [4]:
import pymysql

conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                       database="smartpark_lab", charset="utf8mb4", autocommit=True)
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM houseinfo")
    print("入库后 houseinfo 总条数 :", cur.fetchone()[0])

    cur.execute("SELECT id, title, house, area, price FROM houseinfo ORDER BY id LIMIT 5")
    for row in cur.fetchall():
        print(row)

    cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
    db_set = {tuple(x) for x in cur.fetchall()}
conn.close()

csv_set = {tuple(r[k] for k in FIELDS) for r in rows_csv}
print()
print("CSV 记录数 :", len(csv_set), "| 库内记录数 :", len(db_set))
print("8 个字段内容完全一致 :", csv_set == db_set)
print("CSV 有而库内没有 :", len(csv_set - db_set), "| 库内有而 CSV 没有 :", len(db_set - csv_set))

入库后 houseinfo 总条数 : 498
(1, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '89.9㎡', '16465元/㎡')
(2, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '两代一家', '94.5㎡', '14815元/㎡')
(3, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '大雅云居山', '88㎡', '16487元/㎡')
(4, '中交中央公园 横厅精装大两房 出门都是公园', '中交中央公园悦颂', '91㎡', '20660元/㎡')
(5, '富力现代广场 豪装大两房 户型方正 看房方便', '富力现代广场', '85.5㎡', '16969元/㎡')

CSV 记录数 : 498 | 库内记录数 : 498
8 个字段内容完全一致 : True
CSV 有而库内没有 : 0 | 库内有而 CSV 没有 : 0


### 4. 三条样本逐字段核对（网页 / CSV / 数据库）

第 1 条取自第 1 页，第 61 条取自第 2 页（跨页），第 498 条取自第 10 页。
下面把 CSV 与数据库两份值一起打印，逐字段对照。

In [5]:
import pymysql

conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                       database="smartpark_lab", charset="utf8mb4", autocommit=True)
with conn.cursor() as cur:
    for pos in (0, 60, len(rows_csv) - 1):
        rec = rows_csv[pos]
        cur.execute("SELECT title, house, address, struct, area, floor, maketime, price "
                    "FROM houseinfo WHERE title=%s AND house=%s AND address=%s",
                    (rec["title"], rec["house"], rec["address"]))
        db = cur.fetchone()
        csv_t = tuple(rec[k] for k in FIELDS)
        print(f"\n---- CSV 第 {pos + 1} 条 | 库内命中 {db is not None} | 8 字段一致 {db == csv_t} ----")
        for k, a, b in zip(FIELDS, csv_t, db):
            print(f"  {k:9s} CSV={a!r}")
            print(f"  {'':9s} DB ={b!r}")
conn.close()


---- CSV 第 1 条 | 库内命中 True | 8 字段一致 True ----
  title     CSV='万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
            DB ='万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
  house     CSV='万科金域学府翰林菁华里'
            DB ='万科金域学府翰林菁华里'
  address   CSV='南岸学府大道二塘路59号'
            DB ='南岸学府大道二塘路59号'
  struct    CSV='3室2厅1卫'
            DB ='3室2厅1卫'
  area      CSV='89.9㎡'
            DB ='89.9㎡'
  floor     CSV='低层(共27层)'
            DB ='低层(共27层)'
  maketime  CSV='2019年建造'
            DB ='2019年建造'
  price     CSV='16465元/㎡'
            DB ='16465元/㎡'

---- CSV 第 61 条 | 库内命中 True | 8 字段一致 True ----
  title     CSV='万科物业自营 朝中庭 业主诚心出售'
            DB ='万科物业自营 朝中庭 业主诚心出售'
  house     CSV='万科金域学府翰林菁华里'
            DB ='万科金域学府翰林菁华里'
  address   CSV='南岸学府大道二塘路59号'
            DB ='南岸学府大道二塘路59号'
  struct    CSV='3室2厅1卫'
            DB ='3室2厅1卫'
  area      CSV='85.7㎡'
            DB ='85.7㎡'
  floor     CSV='低层(共27层)'
            DB ='低层(共27层)'
  maketime  CSV='2018年建造'
            DB ='2018年建造'
  price     CSV='16914元/㎡'
  

## 思考题（写进 REPORT.md）

1. 下一页是怎样取得的？结合自己的代码，解释 `href`、`urljoin` 和停止条件。
2. 旧代码把 `out_list` 定义在全局。你把结果列表放在哪里？为什么？
3. 你怎样获取建造年代？节点缺失或顺序改变时，怎样处理？